In [0]:
%run ./common/_utils_config

In [0]:
fq = f"{catalog}.{schema}"

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fq}.data_coverage()
RETURNS TABLE (dataset STRING, first_date DATE, last_date DATE)
COMMENT 'Returns the first and last date available for electricity prices and generation data. Call it first when the user mentions relative dates like today, tomorrow or last month.'
RETURN
  SELECT 'hourly_prices', MIN(price_date), MAX(price_date) FROM {fq}.gold_hourly_prices WHERE pvpc_eur_mwh IS NOT NULL
  UNION ALL
  SELECT 'daily_generation', MIN(generation_date), MAX(generation_date) FROM {fq}.gold_daily_generation
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fq}.day_price_profile(p_date DATE COMMENT 'Date, format YYYY-MM-DD')
RETURNS TABLE (hour INT, pvpc_eur_kwh DOUBLE, spot_eur_kwh DOUBLE, cheapness_rank INT)
COMMENT 'Hourly household electricity price (PVPC, EUR/kWh) and wholesale spot price for one day in Spain, with cheapness_rank (1 = cheapest hour). Use it to find the best hours to run appliances.'
RETURN
  SELECT hour, pvpc_eur_kwh, spot_eur_kwh, CAST(RANK() OVER (ORDER BY pvpc_eur_kwh) AS INT)
  FROM {fq}.gold_hourly_prices
  WHERE price_date = p_date
  ORDER BY hour
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fq}.price_summary(
  p_start DATE COMMENT 'Start date YYYY-MM-DD',
  p_end DATE COMMENT 'End date YYYY-MM-DD, inclusive'
)
RETURNS TABLE (days BIGINT, avg_pvpc_eur_kwh DOUBLE, min_pvpc_eur_kwh DOUBLE, max_pvpc_eur_kwh DOUBLE, avg_spot_eur_kwh DOUBLE)
COMMENT 'Average, minimum and maximum household electricity price (PVPC, EUR/kWh) and average spot price between two dates. Call it twice to compare periods.'
RETURN
  SELECT COUNT(DISTINCT price_date), ROUND(AVG(pvpc_eur_kwh), 4), MIN(pvpc_eur_kwh), MAX(pvpc_eur_kwh), ROUND(AVG(spot_eur_kwh), 4)
  FROM {fq}.gold_hourly_prices
  WHERE price_date BETWEEN p_start AND p_end
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fq}.hourly_average_profile(
  p_start DATE COMMENT 'Start date YYYY-MM-DD',
  p_end DATE COMMENT 'End date YYYY-MM-DD, inclusive'
)
RETURNS TABLE (hour INT, avg_pvpc_eur_kwh DOUBLE)
COMMENT 'Average household electricity price for each hour of the day over a period. Use it to find which hours are usually cheapest or most expensive.'
RETURN
  SELECT hour, ROUND(AVG(pvpc_eur_kwh), 4)
  FROM {fq}.gold_hourly_prices
  WHERE price_date BETWEEN p_start AND p_end
  GROUP BY hour
  ORDER BY hour
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fq}.generation_mix(
  p_start DATE COMMENT 'Start date YYYY-MM-DD',
  p_end DATE COMMENT 'End date YYYY-MM-DD, inclusive'
)
RETURNS TABLE (technology STRING, total_mwh DOUBLE, share_pct DOUBLE)
COMMENT 'Electricity generation in Spain by technology (solar, wind, hydro, nuclear, combined cycle...) between two dates, with share of the total in percent.'
RETURN
  SELECT technology, ROUND(SUM(generation_mwh), 0), ROUND(100 * SUM(generation_mwh) / SUM(SUM(generation_mwh)) OVER (), 1)
  FROM {fq}.gold_daily_generation
  WHERE generation_date BETWEEN p_start AND p_end
  GROUP BY technology
  ORDER BY 2 DESC
""")

In [0]:
display(spark.sql(f"SELECT * FROM {fq}.data_coverage()"))
display(spark.sql(f"SELECT * FROM {fq}.day_price_profile(current_date())"))
display(spark.sql(f"SELECT * FROM {fq}.generation_mix(date_sub(current_date(), 7), current_date())"))